# Static vs Adaptive VC — same-history comparison

This notebook tests whether the **static virtual customer** and the **adaptive virtual customer** produce different next-turn behavior given the **same conversation history**, **same CSR message**, and the same evaluator `state` / `score`.

It uses the same generation path as `POST /chat` in `teleperformance/backend/main.py`: `call_llm(...)` with `training=False`.

| Condition | How `/chat` builds the system prompt |
|-----------|--------------------------------------|
| **Static VC** (`cond1`, `cond2`) | Scenario + response-format only. `feedback` is ignored. |
| **Adaptive VC** (`cond3`, `cond4`) | Same as static, plus a competency prompt loaded from `prompts/competency/{state}_{score}.txt`. |

`state` is one of `Problem Interpretation`, `Problem Exploration`, `Problem Resolution`. `score` is `0`, `1`, or `2`. The filename is `{state.lower().replace(" ", "_")}_{score}.txt` (for example `problem_exploration_0.txt`).

**Output of this experiment:** the virtual customer's next utterance under each condition (`static_vc_response`, `adaptive_vc_response`) — that is what `/chat` returns as `customer_response` — plus two blind LLM judges of that next-turn utterance: **frustration** (0–10) and **CSAT** (customer-expressed satisfaction, 1–5).

Point `DATA_PATH` to the final set of scenarios.

## Experimental setup

Same conversation history and CSR message go to both virtual customers. The only difference is whether a competency prompt is injected. Blind LLM judges then score **frustration** and **CSAT** in the two next-turn replies.

```mermaid
flowchart TD
  case["Shared test case<br/>history · CSR message · scenario · persona<br/>state · score"]

  case --> static["Static VC · cond1"]
  case --> adaptive["Adaptive VC · cond3"]

  static --> sPrompt["System prompt:<br/>scenario + format<br/>feedback ignored"]
  adaptive --> aPrompt["System prompt:<br/>scenario + format<br/>+ competency file from state+score"]

  sPrompt --> sCall["call_llm training=False"]
  aPrompt --> aCall["call_llm training=False"]

  sCall --> sOut["static_vc_response"]
  aCall --> aOut["adaptive_vc_response"]

  sOut --> pair["Paired next-turn replies"]
  aOut --> pair

  pair --> shuffle["Blind A/B shuffle<br/>seeded by case id"]
  shuffle --> judge["LLM judge · temperature 0<br/>frustration 0–10 each"]
  judge --> scores["static score · adaptive score<br/>delta · more_frustrated"]
```


## 1. Setup

Load the backend env and import `call_llm` / `build_system_prompt` so this notebook hits the same code as `/chat`, without needing a live session or Firestore.

In [2]:
from __future__ import annotations

import json
import sys
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv

def resolve_roots() -> tuple[Path, Path]:
    """Find the git repo (backend/) and this experiment folder (validation/VC_validation/)."""
    start = Path.cwd().resolve()
    for path in [start, *start.parents]:
        backend = path / "backend"
        repo = path / "validation" / "VC_validation"
        if (backend / "services").is_dir() and (repo / "data").is_dir():
            return path, repo
        nested_backend = path / "teleperformance" / "backend"
        nested_repo = path / "teleperformance" / "validation" / "VC_validation"
        if (nested_backend / "services").is_dir() and (nested_repo / "data").is_dir():
            return path / "teleperformance", nested_repo
        if path.name == "VC_validation" and (path.parent.parent / "backend" / "services").is_dir():
            return path.parent.parent, path
    raise RuntimeError(f"Could not locate project roots from {start}")


PROJECT_ROOT, REPO_ROOT = resolve_roots()
BACKEND_DIR = PROJECT_ROOT / "backend"
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
DATA_PATH = DATA_DIR / "test_cases.json"

load_dotenv(BACKEND_DIR / ".env")
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from services.llm_service import build_system_prompt, call_llm  # noqa: E402

STATIC_CONDITION = "cond1"    # static VC: competency prompt is not loaded
ADAPTIVE_CONDITION = "cond3"  # adaptive VC: competency prompt is loaded from state+score

VALID_STATES = {
    "Problem Interpretation",
    "Problem Exploration",
    "Problem Resolution",
}
VALID_SCORES = {0, 1, 2}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"backend: {BACKEND_DIR}")
print(f"data path: {DATA_PATH} (exists={DATA_PATH.exists()})")
print(f"static condition: {STATIC_CONDITION}")
print(f"adaptive condition: {ADAPTIVE_CONDITION}")

Using provider: openai
Using model: gpt-4o
backend: /Users/simretgebreegziabher/Documents/Projects/teleperformance/teleperformance/backend
data path: /Users/simretgebreegziabher/Documents/Projects/teleperformance/teleperformance/validation/VC_validation/data/test_cases.json (exists=True)
static condition: cond1
adaptive condition: cond3


## 2. Test-data schema

Each case should look like this. `history` uses the same roles as `/chat`:

- `assistant` = virtual customer
- `user` = CSR

`csr_message` is the latest CSR utterance the VC should reply to (the `message` field on `/chat`). If you omit it, the last `user` turn is taken from `history`.

```json
{
  "id": "case_001",
  "scenario": "flight_cancellation",
  "persona": "angry",
  "history": [
    {"role": "assistant", "content": "...customer opener..."},
    {"role": "user", "content": "...earlier CSR turn..."},
    {"role": "assistant", "content": "...earlier customer turn..."}
  ],
  "csr_message": "The latest CSR response the VC should react to.", //generate CSR responses to the that are approporiate for all scores but sample the history so that the experiment dataset has equal distributions of the 3 states
  "state": "Problem Exploration",
  "score": 0
}
```

Put a JSON **array** of these objects at `VC_validation/data/test_cases.json`. Until that file exists, the cell below uses a placeholder case.

In [3]:
PLACEHOLDER_CASES = [
    {
        "id": "placeholder_flight_exploration_0",
        "scenario": "flight_cancellation",
        "persona": "angry",
        "history": [
            {
                "role": "assistant",
                "content": (
                    "Hi, this is Jordan Blake. My flight UA 4492 from LAX to JFK was cancelled "
                    "because of weather in Chicago, and I need to get to New York today — not tomorrow."
                ),
            }
        ],
        "csr_message": "Can I have your confirmation number please?",
        "state": "Problem Exploration",
        "score": 1,
    }
]


def load_raw_cases(path: Path) -> list[dict]:
    if not path.exists():
        print(f"No test file at {path}. Using {len(PLACEHOLDER_CASES)} placeholder case(s).")
        return deepcopy(PLACEHOLDER_CASES)
    with path.open(encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        data = data.get("cases", data.get("data", [data]))
    if not isinstance(data, list):
        raise ValueError(f"{path} must be a JSON array of cases (or {{{{cases: [...]}}}}).")
    print(f"Loaded {len(data)} case(s) from {path}")
    return data


def normalize_case(raw: dict, index: int) -> dict:
    case = deepcopy(raw)
    case.setdefault("id", f"case_{index:03d}")
    case.setdefault("persona", "angry")

    history = list(case.get("history") or [])
    csr_message = (case.get("csr_message") or "").strip()
    if not csr_message:
        if not history or history[-1].get("role") != "user":
            raise ValueError(
                f"{case['id']}: provide csr_message, or end history with a user (CSR) turn."
            )
        csr_message = history[-1]["content"]
        history = history[:-1]

    state = case.get("state", "")
    score = case.get("score")
    if state not in VALID_STATES:
        raise ValueError(f"{case['id']}: state must be one of {sorted(VALID_STATES)}, got {state!r}")
    if score not in VALID_SCORES:
        raise ValueError(f"{case['id']}: score must be one of {sorted(VALID_SCORES)}, got {score!r}")
    if not case.get("scenario"):
        raise ValueError(f"{case['id']}: scenario is required")

    case["history"] = [{"role": t["role"], "content": t["content"]} for t in history]
    case["csr_message"] = csr_message
    case["score"] = int(score)
    case["competency_file"] = f"{state.lower().replace(' ', '_')}_{int(score)}.txt"
    return case


raw_cases = load_raw_cases(DATA_PATH)
cases = [normalize_case(raw, i) for i, raw in enumerate(raw_cases)]
print(f"Ready: {len(cases)} case(s)")
for case in cases:
    print(
        f"  {case['id']}: scenario={case['scenario']}  "
        f"state={case['state']}  score={case['score']}  "
        f"history_turns={len(case['history'])}  competency={case['competency_file']}"
    )

Loaded 53 case(s) from /Users/simretgebreegziabher/Documents/Projects/teleperformance/VC_validation/data/test_cases.json
Ready: 53 case(s)
  baggage_delay_00__problem_interpretation__score0: scenario=baggage_delay  state=Problem Interpretation  score=0  history_turns=1  competency=problem_interpretation_0.txt
  baggage_delay_00__problem_interpretation__score1: scenario=baggage_delay  state=Problem Interpretation  score=1  history_turns=1  competency=problem_interpretation_1.txt
  baggage_delay_00__problem_interpretation__score2: scenario=baggage_delay  state=Problem Interpretation  score=2  history_turns=1  competency=problem_interpretation_2.txt
  baggage_delay_00__problem_exploration__score0: scenario=baggage_delay  state=Problem Exploration  score=0  history_turns=3  competency=problem_exploration_0.txt
  baggage_delay_00__problem_exploration__score1: scenario=baggage_delay  state=Problem Exploration  score=1  history_turns=3  competency=problem_exploration_1.txt
  baggage_delay_00_

## 3. Confirm the prompt split (no LLM call)

`build_system_prompt` is the only place static vs adaptive diverges. Same scenario / history / CSR message; adaptive additionally injects the competency file named by `state` + `score`.

In [4]:
def prompt_pair(case: dict) -> dict:
    feedback = {"state": case["state"], "score": case["score"]}
    shared = dict(
        scenario=case["scenario"],
        persona=case["persona"],
        training=False,
        feedback=feedback,
    )
    static_prompt = build_system_prompt(**shared, condition=STATIC_CONDITION)
    adaptive_prompt = build_system_prompt(**shared, condition=ADAPTIVE_CONDITION)
    return {
        "id": case["id"],
        "competency_file": case["competency_file"],
        "static_chars": len(static_prompt),
        "adaptive_chars": len(adaptive_prompt),
        "prompts_identical": static_prompt == adaptive_prompt,
        "adaptive_only_chars": len(adaptive_prompt) - len(static_prompt),
        "static_prompt": static_prompt,
        "adaptive_prompt": adaptive_prompt,
    }


prompt_rows = [prompt_pair(case) for case in cases]
for row in prompt_rows:
    print(
        f"{row['id']}: static={row['static_chars']} chars  "
        f"adaptive={row['adaptive_chars']} chars  "
        f"identical={row['prompts_identical']}  "
        f"delta={row['adaptive_only_chars']:+d}  "
        f"file={row['competency_file']}"
    )

# Show the extra adaptive text for the first case
first = prompt_rows[0]
if not first["prompts_identical"]:
    static_p, adaptive_p = first["static_prompt"], first["adaptive_prompt"]
    extra = adaptive_p[len(static_p):] if adaptive_p.startswith(static_p) else None
    print("\n--- adaptive-only competency block (first case) ---")
    print((extra or adaptive_p)[:1500])

DEBUG: competency prompt is problem_interpretation_0.txt
DEBUG: competency prompt is problem_interpretation_1.txt
DEBUG: competency prompt is problem_interpretation_2.txt
DEBUG: competency prompt is problem_exploration_0.txt
DEBUG: competency prompt is problem_exploration_1.txt
DEBUG: competency prompt is problem_exploration_2.txt
DEBUG: competency prompt is problem_resolution_0.txt
DEBUG: competency prompt is problem_resolution_1.txt
DEBUG: competency prompt is problem_resolution_2.txt
DEBUG: competency prompt is problem_interpretation_0.txt
DEBUG: competency prompt is problem_interpretation_1.txt
DEBUG: competency prompt is problem_interpretation_2.txt
DEBUG: competency prompt is problem_exploration_0.txt
DEBUG: competency prompt is problem_exploration_1.txt
DEBUG: competency prompt is problem_exploration_2.txt
DEBUG: competency prompt is problem_resolution_0.txt
DEBUG: competency prompt is problem_resolution_1.txt
DEBUG: competency prompt is problem_resolution_2.txt
DEBUG: competenc

## 4. Generate paired VC responses

Mirrors `/chat`:

```python
call_llm(
    scenario=..., persona=..., training=False,
    message=csr_message, history=history,
    condition=condition, feedback={"state": ..., "score": ...},
)
```

Both calls get the same `feedback` dict. Static still ignores it; adaptive uses it to pick the competency prompt. Set `N_SAMPLES` > 1 for multiple draws (customer temperature is 1.0).

In [5]:
RUN_LLM = True  # set False to skip API calls and only inspect prompts
N_SAMPLES = 1


def generate_pair(case: dict, sample_idx: int = 0) -> dict:
    feedback = {"state": case["state"], "score": case["score"]}
    shared = dict(
        scenario=case["scenario"],
        persona=case["persona"],
        training=False,
        message=case["csr_message"],
        history=case["history"],
        feedback=feedback,
    )
    static = call_llm(**shared, condition=STATIC_CONDITION)
    adaptive = call_llm(**shared, condition=ADAPTIVE_CONDITION)
    static_text = static["customer_response"]
    adaptive_text = adaptive["customer_response"]
    return {
        "id": case["id"],
        "sample": sample_idx,
        "scenario": case["scenario"],
        "persona": case["persona"],
        "state": case["state"],
        "score": case["score"],
        "competency_file": case["competency_file"],
        "history_turns": len(case["history"]),
        "csr_message": case["csr_message"],
        "static_condition": STATIC_CONDITION,
        "adaptive_condition": ADAPTIVE_CONDITION,
        "static_vc_response": static_text,
        "adaptive_vc_response": adaptive_text,
        "responses_identical": static_text.strip() == adaptive_text.strip(),
    }


results: list[dict] = []
if RUN_LLM:
    for case in cases:
        for sample_idx in range(N_SAMPLES):
            row = generate_pair(case, sample_idx)
            results.append(row)
            print(f"\n=== {row['id']}  sample={sample_idx}  identical={row['responses_identical']} ===")
            print(f"CSR: {row['csr_message']}")
            print(f"\n[static / {STATIC_CONDITION}]\n{row['static_vc_response']}")
            print(f"\n[adaptive / {ADAPTIVE_CONDITION} ← {row['competency_file']}]\n{row['adaptive_vc_response']}")
else:
    print("RUN_LLM is False — skipped generation.")

DEBUG: competency prompt is problem_interpretation_0.txt

=== baggage_delay_00__problem_interpretation__score0  sample=0  identical=False ===
CSR: I'm sorry to hear about the inconvenience, Alex. Let's see what we can do to help with your situation.

[static / cond1]
Thank you. I filed the PIR and was told I'd hear something soon, but so far, there's been nothing. I really need to know where my bag is and when I can expect to get it back.

[adaptive / cond3 ← problem_interpretation_0.txt]
I appreciate you looking into it, but what I really need is a specific update on my bag. It's been 48 hours since I filed the report, and I need to know what's being done to resolve this quickly. Can you give me any information on the current status or next steps?
DEBUG: competency prompt is problem_interpretation_1.txt

=== baggage_delay_00__problem_interpretation__score1  sample=0  identical=False ===
CSR: I understand that you filed a report for your missing bag two days ago and haven't received an

## 5. Results table and save

In [6]:
try:
    import pandas as pd

    pd.set_option("display.max_colwidth", 200)
    pd.set_option("display.max_rows", 200)
    results_df = pd.DataFrame(results)
    # display_cols = [
    #     "id", "state", "score", "competency_file", "responses_identical",
    #     "csr_message", "static_vc_response", "adaptive_vc_response",
    # ]
    # display(results_df[display_cols] if not results_df.empty else results_df)
except ImportError:
    results_df = None
    print(json.dumps(results, indent=2, ensure_ascii=False))

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
out_json = OUTPUT_DIR / f"static_vs_adaptive_{stamp}.json"
payload = {
    "static_condition": STATIC_CONDITION,
    "adaptive_condition": ADAPTIVE_CONDITION,
    "n_samples": N_SAMPLES,
    "generated_at": stamp,
    "results": results,
}
out_json.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nSaved {len(results)} row(s) → {out_json}")

if results_df is not None and not results_df.empty:
    out_csv = OUTPUT_DIR / f"static_vs_adaptive_{stamp}.csv"
    results_df.to_csv(out_csv, index=False)
    print(f"Saved CSV → {out_csv}")


Saved 53 row(s) → /Users/simretgebreegziabher/Documents/Projects/teleperformance/VC_validation/outputs/static_vs_adaptive_20260825T182432Z.json
Saved CSV → /Users/simretgebreegziabher/Documents/Projects/teleperformance/VC_validation/outputs/static_vs_adaptive_20260825T182432Z.csv


## 6. LLM judge: frustration level

Score **expressed frustration** in both VC utterances in one LLM call per row. The judge sees the same history and CSR message both replies were generated from, but **not** which reply is static vs adaptive.

Labels `A` / `B` are randomly assigned per row (seeded by case id) so position bias does not systematically favor one condition. Each reply is scored independently on a 0–10 scale; the judge also says which is more frustrated, or `tie`.

Adaptive competency prompts tell the VC to **stay / increase** frustration at score `0` and **ease** it at scores `1`–`2`. The judge is not told that; this cell only measures what was expressed.

In [7]:
import hashlib
import random as rng

from config import MODEL_NAME
from services.llm_service import client

RUN_JUDGE = True  # set False to skip API calls and only inspect generation results
JUDGE_TEMPERATURE = 0.0

FRUSTRATION_JUDGE_SYSTEM = """You are scoring expressed customer frustration in a support conversation.

You will see the conversation so far, the CSR's latest message, and two candidate customer replies (A and B). Score each reply independently. Do not assume one reply is "better" or from a different system.

## Frustration scale (integer 0–10)

0  None — calm, patient, fully cooperative
2  Slight — mild urgency or impatience, still cooperative
4  Mild — clearly annoyed; clipped, insistent, or mildly skeptical
6  Moderate — irritation is the main tone; wants faster progress
8  High — openly frustrated or accusatory; little patience left
10 Extreme — hostile, explosive, or refusing to continue constructively

Odd numbers (1, 3, 5, 7, 9) are allowed for in-between cases.

## Rules

- Score only the frustration / anger / impatience **expressed in that utterance**.
- Use history and the CSR message only as context for what the customer is reacting to.
- Do not reward or penalize how helpful or on-topic the reply is.
- Quote short evidence from the utterance you are scoring.
- Return JSON only.

## Output JSON

{
  "response_a": {
    "frustration_score": <integer 0-10>,
    "label": "<none|slight|mild|moderate|high|extreme>",
    "evidence": "<short quote from A>",
    "rationale": "<1-2 sentences>"
  },
  "response_b": {
    "frustration_score": <integer 0-10>,
    "label": "<none|slight|mild|moderate|high|extreme>",
    "evidence": "<short quote from B>",
    "rationale": "<1-2 sentences>"
  },
  "more_frustrated": "<a|b|tie>",
  "comparison_rationale": "<1 sentence>"
}
"""

LABEL_FROM_SCORE = {
    0: "none",
    1: "slight",
    2: "slight",
    3: "mild",
    4: "mild",
    5: "moderate",
    6: "moderate",
    7: "high",
    8: "high",
    9: "extreme",
    10: "extreme",
}


def _rng_for(case_id: str, sample: int) -> rng.Random:
    seed = int(hashlib.sha256(f"{case_id}:{sample}".encode()).hexdigest()[:8], 16)
    return rng.Random(seed)


def format_history_for_judge(history: list[dict], csr_message: str) -> str:
    lines: list[str] = []
    for turn in history:
        speaker = "Customer" if turn.get("role") == "assistant" else "CSR"
        lines.append(f"{speaker}: {turn.get('content', '')}")
    lines.append(f"CSR: {csr_message}")
    return "\n".join(lines)


def parse_judge_json(raw: str) -> dict:
    text = (raw or "").strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.startswith("json"):
            text = text[4:]
        text = text.strip()
    return json.loads(text)


def clamp_score(value) -> int:
    try:
        score = int(round(float(value)))
    except (TypeError, ValueError):
        return 0
    return max(0, min(10, score))


def judge_pair(row: dict, history: list[dict]) -> dict:
    order = ["static", "adaptive"]
    _rng_for(row["id"], row.get("sample", 0)).shuffle(order)
    labeled = {
        "a": order[0],
        "b": order[1],
    }
    texts = {
        "static": row["static_vc_response"],
        "adaptive": row["adaptive_vc_response"],
    }
    user_payload = (
        "Conversation so far (Customer = virtual customer, CSR = agent):\n\n"
        f"{format_history_for_judge(history, row['csr_message'])}\n\n"
        "Candidate customer replies to that CSR message:\n\n"
        f"Reply A:\n{texts[labeled['a']]}\n\n"
        f"Reply B:\n{texts[labeled['b']]}\n\n"
        "Score Reply A and Reply B independently as instructed."
    )
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": FRUSTRATION_JUDGE_SYSTEM},
            {"role": "user", "content": user_payload},
        ],
        temperature=JUDGE_TEMPERATURE,
        max_tokens=800,
        response_format={"type": "json_object"},
    )
    parsed = parse_judge_json(response.choices[0].message.content)
    by_condition = {}
    for letter, condition in labeled.items():
        block = parsed.get(f"response_{letter}") or {}
        score = clamp_score(block.get("frustration_score"))
        by_condition[condition] = {
            "frustration_score": score,
            "label": block.get("label") or LABEL_FROM_SCORE[score],
            "evidence": block.get("evidence", ""),
            "rationale": block.get("rationale", ""),
        }
    raw_more = str(parsed.get("more_frustrated", "tie")).strip().lower()
    if raw_more in {"a", "b"}:
        more_frustrated = labeled[raw_more]
    else:
        more_frustrated = "tie"
    static_score = by_condition["static"]["frustration_score"]
    adaptive_score = by_condition["adaptive"]["frustration_score"]
    return {
        "judge_model": MODEL_NAME,
        "judge_order": labeled,
        "static_frustration_score": static_score,
        "static_frustration_label": by_condition["static"]["label"],
        "static_frustration_evidence": by_condition["static"]["evidence"],
        "static_frustration_rationale": by_condition["static"]["rationale"],
        "adaptive_frustration_score": adaptive_score,
        "adaptive_frustration_label": by_condition["adaptive"]["label"],
        "adaptive_frustration_evidence": by_condition["adaptive"]["evidence"],
        "adaptive_frustration_rationale": by_condition["adaptive"]["rationale"],
        "frustration_delta_adaptive_minus_static": adaptive_score - static_score,
        "more_frustrated": more_frustrated,
        "comparison_rationale": parsed.get("comparison_rationale", ""),
    }


cases_by_id = {case["id"]: case for case in cases}
judged: list[dict] = []

if not RUN_JUDGE:
    print("RUN_JUDGE is False — skipped frustration scoring.")
elif not results:
    print("No generation results in memory — run section 4 first.")
else:
    for row in results:
        case = cases_by_id.get(row["id"])
        if case is None:
            raise KeyError(f"No matching case for result id {row['id']!r}")
        judgment = judge_pair(row, case["history"])
        scored = {**row, **judgment}
        judged.append(scored)
        delta = scored["frustration_delta_adaptive_minus_static"]
        print(f"\n=== {scored['id']}  sample={scored.get('sample', 0)} ===")
        print(
            f"static   {scored['static_frustration_score']:>2}/10  "
            f"({scored['static_frustration_label']})"
        )
        print(f"  {scored['static_vc_response']}")
        print(
            f"adaptive {scored['adaptive_frustration_score']:>2}/10  "
            f"({scored['adaptive_frustration_label']})"
        )
        print(f"  {scored['adaptive_vc_response']}")
        print(
            f"more frustrated: {scored['more_frustrated']}  "
            f"delta(adaptive-static)={delta:+d}"
        )
        print(scored["comparison_rationale"])


=== baggage_delay_00__problem_interpretation__score0  sample=0 ===
static    6/10  (moderate)
  Thank you. I filed the PIR and was told I'd hear something soon, but so far, there's been nothing. I really need to know where my bag is and when I can expect to get it back.
adaptive  4/10  (mild)
  I appreciate you looking into it, but what I really need is a specific update on my bag. It's been 48 hours since I filed the report, and I need to know what's being done to resolve this quickly. Can you give me any information on the current status or next steps?
more frustrated: static  delta(adaptive-static)=-2
Reply B shows more irritation and urgency compared to Reply A, which still includes some appreciation.

=== baggage_delay_00__problem_interpretation__score1  sample=0 ===
static    2/10  (slight)
  Yes, that's correct. I really need to know what's happening with my bag since it contains important items that I need for tomorrow. Can you please tell me where my bag is or what the next s

In [8]:
if not judged:
    print("No judged rows to display.")
else:
    try:
        import pandas as pd

        pd.set_option("display.max_colwidth", 200)
        pd.set_option("display.max_rows", 200)
        judged_df = pd.DataFrame(judged)
        # display_cols = [
        #     "id", "state", "score",
        #     "static_frustration_score", "adaptive_frustration_score",
        #     "frustration_delta_adaptive_minus_static", "more_frustrated",
        #     "static_vc_response", "adaptive_vc_response",
        # ]
        # display(judged_df[display_cols])
        print(
            "\nMean frustration  "
            f"static={judged_df['static_frustration_score'].mean():.2f}  "
            f"adaptive={judged_df['adaptive_frustration_score'].mean():.2f}  "
            f"delta={judged_df['frustration_delta_adaptive_minus_static'].mean():+.2f}"
        )
        print(judged_df["more_frustrated"].value_counts().to_string())
    except ImportError:
        judged_df = None
        print(json.dumps(judged, indent=2, ensure_ascii=False))

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    out_json = OUTPUT_DIR / f"static_vs_adaptive_frustration_{stamp}.json"
    payload = {
        "static_condition": STATIC_CONDITION,
        "adaptive_condition": ADAPTIVE_CONDITION,
        "judge_model": MODEL_NAME,
        "judge_temperature": JUDGE_TEMPERATURE,
        "n_samples": N_SAMPLES,
        "generated_at": stamp,
        "results": judged,
    }
    out_json.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\nSaved {len(judged)} judged row(s) → {out_json}")

    if judged_df is not None:
        out_csv = OUTPUT_DIR / f"static_vs_adaptive_frustration_{stamp}.csv"
        judged_df.to_csv(out_csv, index=False)
        print(f"Saved CSV → {out_csv}")


Mean frustration  static=2.08  adaptive=2.43  delta=+0.36
more_frustrated
tie         21
adaptive    19
static      13

Saved 53 judged row(s) → /Users/simretgebreegziabher/Documents/Projects/teleperformance/VC_validation/outputs/static_vs_adaptive_frustration_20260825T182804Z.json
Saved CSV → /Users/simretgebreegziabher/Documents/Projects/teleperformance/VC_validation/outputs/static_vs_adaptive_frustration_20260825T182804Z.csv


## 7. LLM judge: CSAT (customer-expressed satisfaction)

Score **customer-expressed satisfaction** in both VC utterances in one LLM call per row — the same last customer message the frustration judge sees. The judge gets the conversation so far and the CSR's latest message as context, but scores only that next-turn reply.

Labels `A` / `B` are randomly assigned per row (seeded by case id, independent of the frustration shuffle). Each reply is scored independently on the 1–5 CSAT scale below.

| Score | Label | Customer in that utterance |
|-------|--------|----------------------------|
| **1** | Extremely dissatisfied | Explicit frustration, anxiety, anger, distrust, distress, or strong disappointment. *“I’m frustrated,” “This is ridiculous.”* |
| **2** | Somewhat dissatisfied | Hesitant, confused, uncomfortable, disappointed, or not fully reassured. *“I’m not sure,” “That’s more expensive than I expected.”* |
| **3** | Neutral or unclear | No clear satisfaction or dissatisfaction; cooperative, polite, or gathering information. *“Okay,” “I see.”* |
| **4** | Mostly satisfied | Positive, comfortable, or accepting, without strong satisfaction. *“Okay, sounds good,” “That works.”* |
| **5** | Extremely satisfied | Explicit strong satisfaction, relief, excitement, gratitude, or positive engagement. *“Perfect,” “Thank you so much.”* |


In [9]:
import hashlib
import random as rng

from config import MODEL_NAME
from services.llm_service import client

RUN_CSAT_JUDGE = True  # set False to skip API calls and only inspect generation results
CSAT_JUDGE_TEMPERATURE = 0.0

CSAT_JUDGE_SYSTEM = """You are scoring customer-expressed satisfaction (CSAT) in a support conversation.

You will see the conversation so far, the CSR's latest message, and two candidate customer replies (A and B). Score each reply independently. Do not assume one reply is "better" or from a different system.

## CSAT scale (integer 1–5)

Customer-expressed satisfaction is how satisfied or dissatisfied the customer appears **in that utterance**, based on the customer's own words, tone, and behavior.

1  Extremely dissatisfied — The customer explicitly expresses frustration, anxiety, anger, distrust, distress, or strong disappointment.
   Examples: "I'm frustrated," "I'm worried," "This is ridiculous," "Nobody helped me."

2  Somewhat dissatisfied — The customer sounds hesitant, confused, uncomfortable, disappointed, or not fully reassured.
   Examples: "I'm not sure," "I need to think about it," "That's more expensive than I expected."

3  Neutral or unclear — The customer does not clearly express satisfaction or dissatisfaction. They may be cooperative, polite, or simply gathering information.
   Examples: "Okay," "I see," "Alright."

4  Mostly satisfied — The customer sounds positive, comfortable, or accepting of the outcome or next step, but does not express strong satisfaction.
   Examples: "Okay, sounds good," "That works," "Great, thank you."

5  Extremely satisfied — The customer explicitly expresses strong satisfaction, relief, excitement, gratitude, or positive engagement.
   Examples: "Perfect," "Awesome," "Thank you so much," "I really appreciate you," "I finally got this done."

## Rules

- Score only the satisfaction / dissatisfaction **expressed in that utterance**.
- Use history and the CSR message only as context for what the customer is reacting to.
- Do not reward or penalize how helpful or on-topic the reply is.
- Quote short evidence from the utterance you are scoring.
- Return JSON only.

## Output JSON

{
  "response_a": {
    "csat_score": <integer 1-5>,
    "label": "<extremely_dissatisfied|somewhat_dissatisfied|neutral_or_unclear|mostly_satisfied|extremely_satisfied>",
    "evidence": "<short quote from A>",
    "rationale": "<1-2 sentences>"
  },
  "response_b": {
    "csat_score": <integer 1-5>,
    "label": "<extremely_dissatisfied|somewhat_dissatisfied|neutral_or_unclear|mostly_satisfied|extremely_satisfied>",
    "evidence": "<short quote from B>",
    "rationale": "<1-2 sentences>"
  },
  "more_satisfied": "<a|b|tie>",
  "comparison_rationale": "<1 sentence>"
}
"""

CSAT_LABEL_FROM_SCORE = {
    1: "extremely_dissatisfied",
    2: "somewhat_dissatisfied",
    3: "neutral_or_unclear",
    4: "mostly_satisfied",
    5: "extremely_satisfied",
}


def _csat_rng(case_id: str, sample: int) -> rng.Random:
    seed = int(hashlib.sha256(f"csat:{case_id}:{sample}".encode()).hexdigest()[:8], 16)
    return rng.Random(seed)


def format_history_for_csat(history: list[dict], csr_message: str) -> str:
    lines: list[str] = []
    for turn in history:
        speaker = "Customer" if turn.get("role") == "assistant" else "CSR"
        lines.append(f"{speaker}: {turn.get('content', '')}")
    lines.append(f"CSR: {csr_message}")
    return "\n".join(lines)


def parse_csat_json(raw: str) -> dict:
    text = (raw or "").strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.startswith("json"):
            text = text[4:]
        text = text.strip()
    return json.loads(text)


def clamp_csat(value) -> int:
    try:
        score = int(round(float(value)))
    except (TypeError, ValueError):
        return 3
    return max(1, min(5, score))


def judge_csat_pair(row: dict, history: list[dict]) -> dict:
    order = ["static", "adaptive"]
    _csat_rng(row["id"], row.get("sample", 0)).shuffle(order)
    labeled = {
        "a": order[0],
        "b": order[1],
    }
    texts = {
        "static": row["static_vc_response"],
        "adaptive": row["adaptive_vc_response"],
    }
    user_payload = (
        "Conversation so far (Customer = virtual customer, CSR = agent):\n\n"
        f"{format_history_for_csat(history, row['csr_message'])}\n\n"
        "Candidate last customer messages in reply to that CSR message:\n\n"
        f"Reply A:\n{texts[labeled['a']]}\n\n"
        f"Reply B:\n{texts[labeled['b']]}\n\n"
        "Score Reply A and Reply B independently as instructed."
    )
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": CSAT_JUDGE_SYSTEM},
            {"role": "user", "content": user_payload},
        ],
        temperature=CSAT_JUDGE_TEMPERATURE,
        max_tokens=800,
        response_format={"type": "json_object"},
    )
    parsed = parse_csat_json(response.choices[0].message.content)
    by_condition = {}
    for letter, condition in labeled.items():
        block = parsed.get(f"response_{letter}") or {}
        score = clamp_csat(block.get("csat_score"))
        by_condition[condition] = {
            "csat_score": score,
            "label": block.get("label") or CSAT_LABEL_FROM_SCORE[score],
            "evidence": block.get("evidence", ""),
            "rationale": block.get("rationale", ""),
        }
    raw_more = str(parsed.get("more_satisfied", "tie")).strip().lower()
    if raw_more in {"a", "b"}:
        more_satisfied = labeled[raw_more]
    else:
        more_satisfied = "tie"
    static_score = by_condition["static"]["csat_score"]
    adaptive_score = by_condition["adaptive"]["csat_score"]
    return {
        "csat_judge_model": MODEL_NAME,
        "csat_judge_order": labeled,
        "static_csat_score": static_score,
        "static_csat_label": by_condition["static"]["label"],
        "static_csat_evidence": by_condition["static"]["evidence"],
        "static_csat_rationale": by_condition["static"]["rationale"],
        "adaptive_csat_score": adaptive_score,
        "adaptive_csat_label": by_condition["adaptive"]["label"],
        "adaptive_csat_evidence": by_condition["adaptive"]["evidence"],
        "adaptive_csat_rationale": by_condition["adaptive"]["rationale"],
        "csat_delta_adaptive_minus_static": adaptive_score - static_score,
        "more_satisfied": more_satisfied,
        "csat_comparison_rationale": parsed.get("comparison_rationale", ""),
    }


csat_source = judged if ("judged" in dir() and judged) else results
csat_judged: list[dict] = []

if not RUN_CSAT_JUDGE:
    print("RUN_CSAT_JUDGE is False — skipped CSAT scoring.")
elif not csat_source:
    print("No generation results in memory — run section 4 first.")
else:
    cases_by_id = {case["id"]: case for case in cases}
    for row in csat_source:
        case = cases_by_id.get(row["id"])
        if case is None:
            raise KeyError(f"No matching case for result id {row['id']!r}")
        judgment = judge_csat_pair(row, case["history"])
        scored = {**row, **judgment}
        csat_judged.append(scored)
        delta = scored["csat_delta_adaptive_minus_static"]
        print(f"\n=== {scored['id']}  sample={scored.get('sample', 0)} ===")
        print(
            f"static   {scored['static_csat_score']}/5  "
            f"({scored['static_csat_label']})"
        )
        print(f"  {scored['static_vc_response']}")
        print(
            f"adaptive {scored['adaptive_csat_score']}/5  "
            f"({scored['adaptive_csat_label']})"
        )
        print(f"  {scored['adaptive_vc_response']}")
        print(
            f"more satisfied: {scored['more_satisfied']}  "
            f"delta(adaptive-static)={delta:+d}"
        )
        print(scored["csat_comparison_rationale"])



=== baggage_delay_00__problem_interpretation__score0  sample=0 ===
static   2/5  (somewhat_dissatisfied)
  Thank you. I filed the PIR and was told I'd hear something soon, but so far, there's been nothing. I really need to know where my bag is and when I can expect to get it back.
adaptive 2/5  (somewhat_dissatisfied)
  I appreciate you looking into it, but what I really need is a specific update on my bag. It's been 48 hours since I filed the report, and I need to know what's being done to resolve this quickly. Can you give me any information on the current status or next steps?
more satisfied: tie  delta(adaptive-static)=+0
Both responses express a similar level of dissatisfaction due to the lack of updates and urgency in resolving the issue.

=== baggage_delay_00__problem_interpretation__score1  sample=0 ===
static   2/5  (somewhat_dissatisfied)
  Yes, that's correct. I really need to know what's happening with my bag since it contains important items that I need for tomorrow. Can 

In [10]:
if not csat_judged:
    print("No CSAT-judged rows to display.")
else:
    try:
        import pandas as pd

        pd.set_option("display.max_colwidth", 200)
        pd.set_option("display.max_rows", 200)
        csat_df = pd.DataFrame(csat_judged)
        # display_cols = [
        #     "id", "state", "score",
        #     "static_csat_score", "adaptive_csat_score",
        #     "csat_delta_adaptive_minus_static", "more_satisfied",
        #     "static_vc_response", "adaptive_vc_response",
        # ]
        # display(csat_df[display_cols])
        print(
            "\nMean CSAT  "
            f"static={csat_df['static_csat_score'].mean():.2f}  "
            f"adaptive={csat_df['adaptive_csat_score'].mean():.2f}  "
            f"delta={csat_df['csat_delta_adaptive_minus_static'].mean():+.2f}"
        )
        print(csat_df["more_satisfied"].value_counts().to_string())
        print("\nBy evaluator score — mean CSAT")
        print(
            csat_df.groupby("score")[["static_csat_score", "adaptive_csat_score"]]
            .mean()
            .round(2)
            .to_string()
        )
    except ImportError:
        csat_df = None
        print(json.dumps(csat_judged, indent=2, ensure_ascii=False))

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    out_json = OUTPUT_DIR / f"static_vs_adaptive_csat_{stamp}.json"
    payload = {
        "static_condition": STATIC_CONDITION,
        "adaptive_condition": ADAPTIVE_CONDITION,
        "judge_model": MODEL_NAME,
        "judge_temperature": CSAT_JUDGE_TEMPERATURE,
        "metric": "csat_customer_expressed_satisfaction",
        "scale": "1-5 on the last customer message only",
        "n_samples": N_SAMPLES,
        "generated_at": stamp,
        "results": csat_judged,
    }
    out_json.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\nSaved {len(csat_judged)} CSAT-judged row(s) → {out_json}")

    if csat_df is not None:
        out_csv = OUTPUT_DIR / f"static_vs_adaptive_csat_{stamp}.csv"
        csat_df.to_csv(out_csv, index=False)
        print(f"Saved CSV → {out_csv}")



Mean CSAT  static=3.08  adaptive=2.92  delta=-0.15
more_satisfied
tie         24
static      17
adaptive    12

By evaluator score — mean CSAT
       static_csat_score  adaptive_csat_score
score                                        
0                   2.94                 2.17
1                   2.94                 2.88
2                   3.33                 3.72

Saved 53 CSAT-judged row(s) → /Users/simretgebreegziabher/Documents/Projects/teleperformance/VC_validation/outputs/static_vs_adaptive_csat_20260825T183445Z.json
Saved CSV → /Users/simretgebreegziabher/Documents/Projects/teleperformance/VC_validation/outputs/static_vs_adaptive_csat_20260825T183445Z.csv


## Result visualizer

Load every JSON under `VC_validation/outputs/` and plot the latest **generation**, **frustration**, and **CSAT** (last-message) runs. Re-run this section after new experiment saves; it does not need the judge cells to be in memory.

In [11]:
# Open all output files and visualize them in the notebook.

from pathlib import Path

import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

try:
    output_dir = OUTPUT_DIR
except NameError:
    output_dir = Path.cwd().resolve()
    if output_dir.name != "VC_validation":
        output_dir = output_dir / "VC_validation"
    output_dir = output_dir / "outputs"

json_paths = sorted(output_dir.glob("static_vs_adaptive*.json"))
if not json_paths:
    raise FileNotFoundError(f"No experiment JSON files in {output_dir}")


def classify_output(path: Path, payload: dict) -> str:
    name = path.name.lower()
    results = payload.get("results") or []
    sample = results[0] if results else {}
    if "csat" in name or "static_csat_score" in sample or "static_csat_last" in sample:
        scale = str(payload.get("scale", ""))
        if "last customer message" in scale or "static_csat_score" in sample:
            return "csat_last_message"
        return "csat_segmented"
    if "frustration" in name or "static_frustration_score" in sample:
        return "frustration"
    return "generation"


catalog_rows = []
loaded: dict[str, dict] = {}
for path in json_paths:
    payload = json.loads(path.read_text(encoding="utf-8"))
    kind = classify_output(path, payload)
    rows = payload.get("results") or []
    loaded[path.stem] = {"path": path, "kind": kind, "payload": payload, "df": pd.DataFrame(rows)}
    catalog_rows.append({
        "file": path.name,
        "kind": kind,
        "generated_at": payload.get("generated_at", ""),
        "n_rows": len(rows),
        "metric": payload.get("metric", ""),
        "scale": payload.get("scale", ""),
        "judge_model": payload.get("judge_model", ""),
    })

catalog = pd.DataFrame(catalog_rows).sort_values(["kind", "generated_at"], ascending=[True, False])
print(f"Loaded {len(catalog)} file(s) from {output_dir}\n")
display(catalog)

latest_by_kind: dict[str, dict] = {}
for kind, group in catalog.sort_values("generated_at", ascending=False).groupby("kind", sort=False):
    stem = Path(group.iloc[0]["file"]).stem
    latest_by_kind[kind] = loaded[stem]
    print(f"latest {kind}: {group.iloc[0]['file']}")

frust_df = latest_by_kind.get("frustration", {}).get("df", pd.DataFrame())
csat_df = latest_by_kind.get("csat_last_message", {}).get("df", pd.DataFrame())
gen_df = latest_by_kind.get("generation", {}).get("df", pd.DataFrame())

# Prefer the CSAT file when it already includes frustration scores from the prior judge.
if not csat_df.empty and "static_frustration_score" in csat_df.columns:
    viz_df = csat_df.copy()
    viz_source = latest_by_kind["csat_last_message"]["path"].name
elif not frust_df.empty:
    viz_df = frust_df.copy()
    viz_source = latest_by_kind["frustration"]["path"].name
    if not csat_df.empty:
        csat_cols = [c for c in csat_df.columns if "csat" in c or c in {"id", "sample"}]
        viz_df = viz_df.merge(csat_df[csat_cols], on=["id", "sample"], how="left")
        viz_source = f"{viz_source} + {latest_by_kind['csat_last_message']['path'].name}"
else:
    viz_df = gen_df.copy()
    viz_source = latest_by_kind.get("generation", {}).get("path")
    viz_source = viz_source.name if viz_source else ""

print(f"\nPlotting {len(viz_df)} row(s) from {viz_source}")
if not gen_df.empty:
    identical_rate = gen_df["responses_identical"].mean() if "responses_identical" in gen_df.columns else float("nan")
    print(f"generation: {len(gen_df)} rows, identical replies={identical_rate:.0%}")



Loaded 3 file(s) from /Users/simretgebreegziabher/Documents/Projects/teleperformance/VC_validation/outputs



,file,kind,generated_at,n_rows,metric,scale,judge_model
1,static_vs_adaptive_csat_20260825T183445Z.json,csat_last_message,20260825T183445Z,53,csat_customer_expressed_satisfaction,1-5 on the last customer message only,gpt-4o
2,static_vs_adaptive_frustration_20260825T182804Z.json,frustration,20260825T182804Z,53,,,gpt-4o
0,static_vs_adaptive_20260825T182432Z.json,generation,20260825T182432Z,53,,,


latest csat_last_message: static_vs_adaptive_csat_20260825T183445Z.json
latest frustration: static_vs_adaptive_frustration_20260825T182804Z.json
latest generation: static_vs_adaptive_20260825T182432Z.json

Plotting 53 row(s) from static_vs_adaptive_csat_20260825T183445Z.json
generation: 53 rows, identical replies=0%


In [18]:
STATE_ORDER = ["Problem Interpretation", "Problem Exploration", "Problem Resolution"]
SCORE_ORDER = [0, 1, 2]
CONDITION_COLORS = {"static": "#636EFA", "adaptive": "#EF553B"}


def show_fig(fig):
    """Render Plotly in the notebook without requiring nbformat mime rendering."""
    from IPython.display import HTML, display as ipy_display

    ipy_display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))


def _condition_long(df: pd.DataFrame, static_col: str, adaptive_col: str, value_name: str) -> pd.DataFrame:
    long = df.melt(
        id_vars=[c for c in ["id", "scenario", "state", "score"] if c in df.columns],
        value_vars=[static_col, adaptive_col],
        var_name="condition",
        value_name=value_name,
    )
    long["condition"] = long["condition"].map({static_col: "static", adaptive_col: "adaptive"})
    if "state" in long.columns:
        long["state"] = pd.Categorical(long["state"], categories=STATE_ORDER, ordered=True)
    if "score" in long.columns:
        long["score"] = pd.Categorical(long["score"], categories=SCORE_ORDER, ordered=True)
    return long


def _mean_bar(long: pd.DataFrame, value: str, group: str, title: str, yaxis: str):
    summary = (
        long.groupby([group, "condition"], observed=True)[value]
        .agg(mean="mean", std="std")
        .reset_index()
        .sort_values(group)
    )
    summary["std"] = summary["std"].fillna(0)
    summary["label"] = [
        f"{mean:.2f} ± {std:.2f}" for mean, std in zip(summary["mean"], summary["std"])
    ]
    fig = px.bar(
        summary,
        x=group,
        y="mean",
        color="condition",
        error_y="std",
        barmode="group",
        color_discrete_map=CONDITION_COLORS,
        title=title,
        labels={"mean": yaxis, group: group.replace("_", " ").title(), "condition": "Condition"},
        template="plotly_white",
        text="label",
    )
    fig.update_traces(textposition="outside")
    ymax = float((summary["mean"] + summary["std"]).max())
    fig.update_layout(yaxis_title=yaxis, legend_title=None, bargap=0.25)
    fig.update_yaxes(range=[0, ymax * 1.25 if ymax > 0 else 1])
    return fig


if viz_df.empty:
    print("No judged rows to plot.")
else:
    overview_rows = []
    if "static_frustration_score" in viz_df.columns:
        overview_rows.append({
            "metric": "Frustration (0–10)",
            "static_mean": viz_df["static_frustration_score"].mean(),
            "static_std": viz_df["static_frustration_score"].std(),
            "adaptive_mean": viz_df["adaptive_frustration_score"].mean(),
            "adaptive_std": viz_df["adaptive_frustration_score"].std(),
            "delta_adaptive_minus_static": viz_df["frustration_delta_adaptive_minus_static"].mean(),
        })
    if "static_csat_score" in viz_df.columns:
        overview_rows.append({
            "metric": "CSAT (1–5, last customer message)",
            "static_mean": viz_df["static_csat_score"].mean(),
            "static_std": viz_df["static_csat_score"].std(),
            "adaptive_mean": viz_df["adaptive_csat_score"].mean(),
            "adaptive_std": viz_df["adaptive_csat_score"].std(),
            "delta_adaptive_minus_static": viz_df["csat_delta_adaptive_minus_static"].mean(),
        })
    overview = pd.DataFrame(overview_rows)
    if not overview.empty:
        display(overview.round(2))


,metric,static_mean,static_std,adaptive_mean,adaptive_std,delta_adaptive_minus_static
0,Frustration (0–10),2.08,1.57,2.43,1.93,0.36
1,"CSAT (1–5, last customer message)",3.08,1.00,2.92,1.12,-0.15


In [19]:
if viz_df.empty:
    print("No judged rows to plot.")
else:
    metrics = [
        {
            "key": "frustration",
            "static_col": "static_frustration_score",
            "adaptive_col": "adaptive_frustration_score",
            "winner_col": "more_frustrated",
            "value_name": "frustration",
            "yaxis": "Frustration (0–10)",
            "score_title": "Mean frustration by evaluator score",
            "state_title": "Mean frustration by conversation state",
            "winner_title": "Who was more frustrated?",
            "paired_title": "Paired frustration: adaptive vs static",
            "x_label": "Static frustration (0–10)",
            "y_label": "Adaptive frustration (0–10)",
            "equal": (0, 10),
            "axis_range": [-0.5, 10.5],
        },
        {
            "key": "csat",
            "static_col": "static_csat_score",
            "adaptive_col": "adaptive_csat_score",
            "winner_col": "more_satisfied",
            "value_name": "csat",
            "yaxis": "CSAT (1–5)",
            "score_title": "Mean CSAT by evaluator score (last customer message)",
            "state_title": "Mean CSAT by conversation state (last customer message)",
            "winner_title": "Who was more satisfied?",
            "paired_title": "Paired CSAT: adaptive vs static",
            "x_label": "Static CSAT (1–5)",
            "y_label": "Adaptive CSAT (1–5)",
            "equal": (1, 5),
            "axis_range": [0.5, 5.5],
        },
    ]

    for metric in metrics:
        if metric["static_col"] not in viz_df.columns:
            print(f"Skipping {metric['key']}: {metric['static_col']} not in data.")
            continue

        long = _condition_long(
            viz_df, metric["static_col"], metric["adaptive_col"], metric["value_name"]
        )
        show_fig(_mean_bar(long, metric["value_name"], "score", metric["score_title"], metric["yaxis"]))
        show_fig(_mean_bar(long, metric["value_name"], "state", metric["state_title"], metric["yaxis"]))

        win = (
            viz_df[metric["winner_col"]]
            .value_counts()
            .reindex(["adaptive", "static", "tie"])
            .fillna(0)
            .reset_index()
        )
        win.columns = ["who", "n"]
        show_fig(px.bar(
            win, x="who", y="n", color="who",
            color_discrete_map={**CONDITION_COLORS, "tie": "#AB63FA"},
            title=metric["winner_title"],
            template="plotly_white",
            text_auto=True,
        ).update_layout(showlegend=False, yaxis_title="Cases"))

        fig = px.scatter(
            viz_df,
            x=metric["static_col"],
            y=metric["adaptive_col"],
            color="score",
            hover_data=["id", "state", "scenario"],
            title=metric["paired_title"],
            labels={
                metric["static_col"]: metric["x_label"],
                metric["adaptive_col"]: metric["y_label"],
                "score": "Evaluator score",
            },
            template="plotly_white",
        )
        lo, hi = metric["equal"]
        fig.add_trace(go.Scatter(
            x=[lo, hi], y=[lo, hi], mode="lines",
            name="equal", line=dict(color="gray", dash="dash"),
        ))
        fig.update_xaxes(range=metric["axis_range"])
        fig.update_yaxes(range=metric["axis_range"], scaleanchor="x", scaleratio=1)
        show_fig(fig)

    if {"static_frustration_score", "static_csat_score"} <= set(viz_df.columns):
        both = pd.concat(
            [
                viz_df.assign(condition="static")[
                    ["condition", "score", "state", "static_frustration_score", "static_csat_score"]
                ].rename(columns={
                    "static_frustration_score": "frustration",
                    "static_csat_score": "csat",
                }),
                viz_df.assign(condition="adaptive")[
                    ["condition", "score", "state", "adaptive_frustration_score", "adaptive_csat_score"]
                ].rename(columns={
                    "adaptive_frustration_score": "frustration",
                    "adaptive_csat_score": "csat",
                }),
            ],
            ignore_index=True,
        )
        show_fig(px.scatter(
            both,
            x="frustration",
            y="csat",
            color="condition",
            facet_col="score",
            color_discrete_map=CONDITION_COLORS,
            hover_data=["state"],
            title="CSAT vs frustration by evaluator score",
            labels={
                "frustration": "Frustration (0–10)",
                "csat": "CSAT (1–5)",
                "score": "Evaluator score",
            },
            template="plotly_white",
        ).update_yaxes(range=[0.5, 5.5]).update_xaxes(range=[-0.5, 10.5]))
